# CNN Challenge: tablero docente

Consultá manualmente los envíos de los alumnos a Google Sheets. La pestaña `runs` conserva **una fila por intento**, con métricas por época y configuración. El botón **Actualizar resultados** vuelve a leer la hoja; no se necesita refresco automático.

El ranking se ordena por `best_val_acc`. Se aceptan cantidades positivas de filtros y épocas; la tabla muestra `epochs` para distinguir presupuestos de entrenamiento. Para comparar arquitecturas, usá la misma cantidad de épocas. El conjunto de test se reserva para evaluar al final una arquitectura seleccionada. Las comparaciones entre cohortes son válidas cuando comparten el mismo `protocolo`.

## 1. Preparar Google Sheets y Apps Script

    1. Creá una hoja de cálculo de Google con una pestaña llamada `runs`. Copiá el ID de la URL (entre `/d/` y `/edit`).
    2. En **Extensiones > Apps Script**, pegá el código siguiente en `Code.gs`.
    3. En **Configuración del proyecto > Propiedades de la secuencia de comandos**, creá `SHEET_ID`, `COHORTE_ACTIVA` (por ejemplo `2026_2`) y `TOKEN_CLASE` (valor largo y aleatorio). No guardes el token en los notebooks.
    4. Desplegá como **aplicación web**, con ejecución **como el propietario** y acceso **cualquier persona**. Compartí con los alumnos la URL que termina en `/exec`, el token y la cohorte activa durante la clase.
    5. Iniciá sesión en el Colab docente con acceso a la hoja para consultar los resultados. Rotá el token entre cohortes; la cohorte activa se valida en el servidor.

    La validación evita errores corrientes y duplicados de `run_id`. Un token compartido no autentica individualmente a cada alumno ni impide que alguien con el token envíe datos inventados; tratá el ranking como una actividad didáctica.

## Código para `Code.gs`

```javascript
function responder(obj) {
  return ContentService.createTextOutput(JSON.stringify(obj))
    .setMimeType(ContentService.MimeType.JSON);
}

function doPost(e) {
  var lock = LockService.getScriptLock();
  try {
    var p = JSON.parse(e.postData.contents);
    var props = PropertiesService.getScriptProperties();
    var token = props.getProperty('TOKEN_CLASE');
    var cohorte = props.getProperty('COHORTE_ACTIVA');
    if (!token || !cohorte || !props.getProperty('SHEET_ID')) {
      throw Error('Faltan propiedades del servidor');
    }
    if (!p || p.token !== token) throw Error('Token inválido');
    if (p.cohorte !== cohorte) throw Error('Cohorte inválida');
    if (p.protocolo !== 'CIFAR10_CNN_v1') throw Error('Protocolo inválido');
    if (!Number.isInteger(p.epochs) || p.epochs < 1) throw Error('Cantidad de épocas inválida');
    if (!/^[\p{L}\p{N}][\p{L}\p{N} _.-]{1,59}$/u.test(p.alumno)) throw Error('Alias inválido');
    if (!/^[0-9a-f-]{36}$/i.test(p.run_id) || !/^[0-9a-f]{32}$/i.test(p.sesion_id)) {
      throw Error('Identificadores inválidos');
    }
    if (!Number.isInteger(p.n_parametros) || p.n_parametros < 1 || p.n_parametros > 1000000) {
      throw Error('Cantidad de parámetros inválida');
    }
    if (!Number.isFinite(p.training_time) || p.training_time <= 0) throw Error('Tiempo inválido');
    var a = p.arquitectura;
    if (!a || !Array.isArray(a.filtros) || a.filtros.length < 1 || a.filtros.length > 4 ||
        !a.filtros.every(x => Number.isInteger(x) && x >= 1) ||
        !Number.isInteger(a.kernel_size) || a.kernel_size < 1 || typeof a.usar_batchnorm !== 'boolean' ||
        !Number.isFinite(a.dropout) || a.dropout < 0 || a.dropout > 0.6 ||
        !['max', 'avg'].includes(a.pooling)) throw Error('Arquitectura inválida');
    var keys = ['train_loss', 'val_loss', 'train_acc', 'val_acc'];
    keys.forEach(function(k) {
      if (!Array.isArray(p[k]) || p[k].length !== p.epochs ||
          !p[k].every(x => typeof x === 'number' && Number.isFinite(x) &&
                       x >= 0 && (k.endsWith('acc') ? x <= 1 : x <= 100))) {
        throw Error('Historial inválido: ' + k);
      }
    });
    if (Math.abs(p.best_val_acc - Math.max.apply(null, p.val_acc)) > 1e-8 ||
        Math.abs(p.final_val_acc - p.val_acc[p.epochs - 1]) > 1e-8) throw Error('Resumen inconsistente');
    if (!lock.tryLock(10000)) throw Error('Servidor ocupado; reintentá el envío');
    var sheet = SpreadsheetApp.openById(props.getProperty('SHEET_ID')).getSheetByName('runs');
    if (!sheet) throw Error('Falta la pestaña runs');
    var headers = ['cohorte', 'alumno', 'run_id', 'sesion_id', 'timestamp', 'protocolo',
      'arquitectura', 'n_parametros', 'epochs', 'train_loss', 'val_loss',
      'train_acc', 'val_acc', 'best_val_acc', 'final_val_acc', 'training_time'];
    if (sheet.getLastRow() === 0) sheet.appendRow(headers);
    if (sheet.getLastRow() > 1) {
      var found = sheet.getRange(2, 3, sheet.getLastRow() - 1, 1)
        .createTextFinder(p.run_id).matchEntireCell(true).findNext();
      if (found) return responder({ok: true, run_id: p.run_id, duplicado: true});
    }
    sheet.appendRow([p.cohorte, p.alumno, p.run_id, p.sesion_id, new Date().toISOString(),
      p.protocolo, JSON.stringify(a), p.n_parametros, p.epochs,
      JSON.stringify(p.train_loss), JSON.stringify(p.val_loss),
      JSON.stringify(p.train_acc), JSON.stringify(p.val_acc),
      p.best_val_acc, p.final_val_acc, p.training_time]);
    return responder({ok: true, run_id: p.run_id, duplicado: false});
  } catch (err) {
    return responder({ok: false, error: String(err.message)});
  } finally {
    if (lock.hasLock()) lock.releaseLock();
  }
}
```

## 2. Conectar este Colab con la hoja

    La autenticación se realiza con tu cuenta de Google. El notebook docente lee la hoja directamente; no necesita el token de envío.

In [1]:
%pip -q install gspread ipywidgets

import json
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import pandas as pd
import gspread
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import auth, output
import google.auth

output.enable_custom_widget_manager()
auth.authenticate_user()
credenciales, _ = google.auth.default()
cliente = gspread.authorize(credenciales)

SHEET_ID = "1rEODM5Td3yWx-7dhZ2fP3xRIuFyeKk0MngB0aCaqfkU"  # ID de la hoja de cálculo del docente.
hoja = cliente.open_by_key(SHEET_ID).worksheet("runs")


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'google.colab'

Deployment ID

AKfycbzcj5rreCM71dcN3MFyFB191Mk36ugSFyYId93DhNqat2N7ItSSdn2QvZ6k2UQiLwaqIQ

URL

https://script.google.com/macros/s/AKfycbzcj5rreCM71dcN3MFyFB191Mk36ugSFyYId93DhNqat2N7ItSSdn2QvZ6k2UQiLwaqIQ/exec

COHORTE_ACTIVA

AP_Co27

token_clase

192837645

## 3. Leer y validar resultados

    Se omiten filas incompletas o de otro protocolo y se muestran avisos para revisarlas. La vista **mejor intento** conserva el `run_id` que logró mayor `best_val_acc` por alumno y cohorte.

In [ ]:
PROTOCOLO = "CIFAR10_CNN_v1"
HISTORIAS = ("train_loss", "val_loss", "train_acc", "val_acc")

def cargar_resultados():
    filas = hoja.get_all_records()
    validas, errores = [], []
    for numero, fila in enumerate(filas, start=2):
        try:
            if fila["protocolo"] != PROTOCOLO:
                continue
            registro = dict(fila)
            registro["arquitectura"] = json.loads(fila["arquitectura"])
            epocas = float(fila["epochs"])
            if not epocas.is_integer() or epocas < 1:
                raise ValueError("epochs debe ser un entero positivo")
            registro["epochs"] = int(epocas)
            for clave in HISTORIAS:
                registro[clave] = json.loads(fila[clave])
                if not isinstance(registro[clave], list) or len(registro[clave]) != registro["epochs"]:
                    raise ValueError(f"{clave} no coincide con epochs")
            for clave in ("best_val_acc", "final_val_acc", "training_time"):
                registro[clave] = float(fila[clave])
            registro["n_parametros"] = int(fila["n_parametros"])
            validas.append(registro)
        except (KeyError, ValueError, TypeError) as exc:
            errores.append(f"Fila {numero}: {exc}")
    return pd.DataFrame(validas), errores

def mejores_por_alumno(df):
    if df.empty:
        return df
    return (df.sort_values(["best_val_acc", "timestamp"], ascending=[False, True])
              .drop_duplicates(["cohorte", "alumno"], keep="first"))

## 4. Dashboard

    Elegí cohorte y modo de vista. Para mantener legibles las curvas, seleccioná hasta cinco corridas en el selector. La tabla muestra el ranking, el gráfico de puntos relaciona accuracy y parámetros, y la brecha `train_acc - val_acc` señala **posible** sobreajuste (no es un diagnóstico definitivo).

In [ ]:
boton_actualizar = widgets.Button(description="Actualizar resultados", button_style="primary")
filtro_cohorte = widgets.Dropdown(description="Cohorte", options=[("Todas", "Todas")])
filtro_modo = widgets.ToggleButtons(description="Intentos", options=[("Todos", "Todos"), ("Mejor por alumno", "Mejor")])
selector_corridas = widgets.SelectMultiple(description="Curvas", options=[], rows=8,
                                            layout=widgets.Layout(width="95%"))
salida = widgets.Output()
registros = pd.DataFrame()

def vista_filtrada():
    df = registros
    if df.empty:
        return df
    if filtro_cohorte.value != "Todas":
        df = df[df["cohorte"] == filtro_cohorte.value]
    if filtro_modo.value == "Mejor":
        df = mejores_por_alumno(df)
    return df.sort_values("best_val_acc", ascending=False)

def actualizar_selector():
    df = vista_filtrada()
    opciones = [(f'{r.cohorte} | {r.alumno} | {r.best_val_acc:.3f} | {r.run_id[:8]}', r.run_id)
                for r in df.itertuples()]
    selector_corridas.options = opciones
    selector_corridas.value = tuple(valor for _, valor in opciones[:5])

def dibujar(_=None):
    with salida:
        clear_output(wait=True)
        df = vista_filtrada()
        if df.empty:
            print("No hay corridas para estos filtros. Pulsá Actualizar resultados.")
            return
        tabla = df[["cohorte", "alumno", "run_id", "arquitectura", "best_val_acc",
                    "final_val_acc", "epochs", "n_parametros", "training_time"]].copy()
        tabla["arquitectura"] = tabla["arquitectura"].map(
            lambda a: f'filtros={a["filtros"]}, k={a["kernel_size"]}, '
                      f'BN={a["usar_batchnorm"]}, dropout={a["dropout"]}, pool={a["pooling"]}'
        )
        tabla.insert(0, "puesto", range(1, len(tabla) + 1))
        tabla_visible = tabla.head(20)
        elegidas = df[df["run_id"].isin(selector_corridas.value)].head(5)
        ids_coloreados = pd.concat([tabla_visible["run_id"], elegidas["run_id"]]).unique()
        tab20 = plt.get_cmap("tab20").colors
        paleta = tab20[::2] + tab20[1::2] + plt.get_cmap("Dark2").colors
        colores = {run_id: to_hex(color) for run_id, color in zip(ids_coloreados, paleta)}
        bordes = tabla_visible["run_id"].map(lambda run_id: f"border-left: 5px solid {colores[run_id]}")
        display(tabla_visible.style.apply(lambda _: bordes, subset="puesto")
                .format({"run_id": lambda valor: valor[:8], "best_val_acc": "{:.3f}",
                         "final_val_acc": "{:.3f}", "training_time": "{:.1f}"}))
        fig, axes = plt.subplots(1, 3, figsize=(17, 4))
        for r in elegidas.itertuples():
            nombre = f"{r.alumno} ({r.run_id[:8]})"
            epocas = range(1, r.epochs + 1)
            color = colores[r.run_id]
            axes[0].plot(epocas, r.train_loss, "--", color=color, label=nombre + " train")
            axes[0].plot(epocas, r.val_loss, color=color, label=nombre + " val")
            axes[1].plot(epocas, r.train_acc, "--", color=color, label=nombre + " train")
            axes[1].plot(epocas, r.val_acc, color=color, label=nombre + " val")
        axes[0].set(title="Loss", xlabel="Epoch")
        axes[1].set(title="Accuracy", xlabel="Epoch")
        colores_scatter = df["run_id"].map(colores).fillna("#b0b0b0")
        axes[2].scatter(df["n_parametros"], df["best_val_acc"], c=colores_scatter)
        axes[2].set(title="Accuracy vs parámetros", xlabel="Parámetros", ylabel="Best val acc")
        for ax in axes:
            ax.grid(alpha=0.25)
        for ax in axes[:2]:
            if len(elegidas):
                ax.legend(fontsize=7)
        fig.tight_layout()
        plt.show()

        brechas = df.copy()
        brechas["brecha_acc"] = brechas["train_acc"].map(lambda v: v[-1]) - brechas["final_val_acc"]
        sospechosos = brechas[brechas["brecha_acc"] >= 0.10]
        if len(sospechosos):
            print("Posible sobreajuste (brecha final >= 0.10):")
            display(sospechosos[["alumno", "cohorte", "run_id", "brecha_acc"]].head(15)
                    .style.format({"run_id": lambda valor: valor[:8]}))
        historico = mejores_por_alumno(registros)
        print("Mejor resultado histórico por cohorte:")
        display(historico.loc[historico.groupby("cohorte")["best_val_acc"].idxmax(),
                             ["cohorte", "alumno", "best_val_acc", "run_id"]]
                .style.format({"run_id": lambda valor: valor[:8]}))

def refrescar(_=None):
    global registros
    try:
        registros, errores = cargar_resultados()
    except Exception as exc:
        with salida:
            clear_output(wait=True)
            print(f"No se pudo leer Google Sheets: {exc}")
        return
    seleccion = filtro_cohorte.value
    cohortes = sorted(registros["cohorte"].unique()) if not registros.empty else []
    filtro_cohorte.options = [("Todas", "Todas")] + [(x, x) for x in cohortes]
    filtro_cohorte.value = seleccion if seleccion in cohortes else "Todas"
    actualizar_selector()
    dibujar()
    if errores:
        with salida:
            print("Filas omitidas:", "; ".join(errores[:10]))

def cambiar_filtro(change):
    if change["name"] == "value":
        actualizar_selector()
        dibujar()

boton_actualizar.on_click(refrescar)
filtro_cohorte.observe(cambiar_filtro, names="value")
filtro_modo.observe(cambiar_filtro, names="value")
selector_corridas.observe(lambda change: dibujar() if change["name"] == "value" else None, names="value")
display(widgets.VBox([widgets.HBox([boton_actualizar, filtro_cohorte, filtro_modo]),
                      selector_corridas, salida]))
refrescar()